In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle
import pandas as pd
from dotenv import load_dotenv
from Code.Utils.util_methods import UtilMethods
from sklearn.model_selection import train_test_split
import numpy as np
#from jupyter_datatables import init_datatables_mode

base = UtilMethods.find_project_root(os.getcwd())
print(f"Project root found: {base}")

if load_dotenv(f'{base}/.env'):
    print(".env found")
else:
    print("ERROR .env not found")

In [ ]:
type = 'Curve' # Curve or Lab
light = 'FL2' # FL2, D65, Studio LED for Lab values
split = 'PrCa_filename' # 'PrCa_filename' if splitting based on the process card and 'random' if random split
validation_dataset = True # Flag if validation dataset is needed
remove_unused_pigments = False # Removes pigments that are not used in any recipes. Keep True for NN, but False for GA
remove_tolerance = 3 # If remove_unused_pigments Ture, removes pigments that are used in remove_tolerance recipes or less

## Load the data

In [ ]:
df = pd.read_pickle(f"{base}/Dataset/input_recipes.pkl")
df

In [ ]:
#list(df.columns)[:]

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.stats import norm

# get distribution of counts
counts = df['PrCa_filename'].value_counts()
distribution = counts.value_counts()

# prepare data for histogram
data = []
for count_val, freq in distribution.items():
    data.extend([count_val] * freq)
data = np.array(data)

# fit a normal distribution to the data
mu, std = norm.fit(data)

# plot histogram
plt.figure(figsize=(6, 4))
plt.hist(data, bins=len(set(data)), alpha=0.6, color='g', density=True)

# plot the fitted gaussian curve
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2)

plt.xlabel('count value')
plt.ylabel('density')
plt.title(f'Histogram of count values with Gaussian fit\nMean={mu:.2f}, Std={std:.2f}')
plt.show()

In [ ]:
reflection_columns = [f'{w}nm' for w in np.arange(400,741,10)]
lab_columns = [f'L_{light}', f'a_{light}', f'b_{light}']
pigment_columns = pd.read_pickle(f"{base}/Dataset/Arnold/used_columns_16April2025.pkl")[:-3]
pigment_columns

## Select the X and y data

In [ ]:
if type == 'Lab':
    X = df[lab_columns]
elif type == 'Curve':
    X = df[reflection_columns]
X

In [ ]:
y = df[pigment_columns]
print('Full shape of y:')
print(y.shape)
if remove_unused_pigments:
    y = y.loc[:, (y > 0).sum() > remove_tolerance]
pigment_columns = list(y.columns)
y

Split to train and test set

In [ ]:
from sklearn.model_selection import GroupShuffleSplit


if split == 'random':
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    if validation_dataset:
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
elif split == 'PrCa_filename':

    group_col = 'PrCa_filename'

    # create the splitter
    splitter = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)

    # run the split to obtain train and test sets
    train_idx, test_idx = next(splitter.split(df, groups=df[group_col]))

    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]

    if validation_dataset:

        # run the split to obtain train and val sets from train set
        train_idx, val_idx = next(splitter.split(train_df, groups=train_df[group_col]))

        val_df = train_df.iloc[val_idx]
        train_df = train_df.iloc[train_idx]


    if type == 'Lab':
        X_train = train_df[lab_columns]
        X_test = test_df[lab_columns]
        if validation_dataset: X_val = val_df[lab_columns]
    elif type == 'Curve':
        X_train = train_df[reflection_columns]
        X_test = test_df[reflection_columns]
        if validation_dataset: X_val = val_df[reflection_columns]

    y_train = train_df[pigment_columns]
    y_test = test_df[pigment_columns]
    if validation_dataset: y_val = val_df[pigment_columns]

    if validation_dataset:
        prc_overlap = set(train_df[group_col]) & set(test_df[group_col]) & set(val_df[group_col])
    else:
        prc_overlap = set(train_df[group_col]) & set(test_df[group_col])
    if len(prc_overlap)==0:
        print('SPLIT BY PROCESS CARD SUCCESFUL')
    else:
        raise Exception('SPLIT BY PROCESS CARD FAILED')

In [ ]:
print(X_train.shape)
X_train

In [ ]:
print(y_train.shape)
y_train

In [ ]:
print(X_test.shape)
X_test

In [ ]:
print(y_test.shape)
y_test

In [ ]:
print(X_val.shape)
X_val

In [ ]:
print(y_val.shape)
y_val

## Save the data

In [ ]:
path = f'{base}/Dataset/traintest'
X.to_csv(f'{path}/X.csv', index=False)
print(f'X saved to {path}')
y.to_csv(f'{path}/y.csv', index=False)
print(f'y saved to {path}')
X_train.to_csv(f'{path}/X_train.csv', index=False)
print(f'X_train saved to {path}')
X_test.to_csv(f'{path}/X_test.csv', index=False)
print(f'X_test saved to {path}')
y_train.to_csv(f'{path}/y_train.csv', index=False)
print(f'y_train saved to {path}')
y_test.to_csv(f'{path}/y_test.csv', index=False)
print(f'y_test saved to {path}')
if validation_dataset:
    X_val.to_csv(f'{path}/X_val.csv', index=False)
    print(f'X_val saved to {path}')
    y_val.to_csv(f'{path}/y_val.csv', index=False)
    print(f'y_val saved to {path}')